# GeoChat on a free Colab T4

Loads [GeoChat-7B](https://huggingface.co/MBZUAI/geochat-7B) in 4-bit, answers one VQA question, runs one grounding query, and prints the **raw** box output so we know exactly what to parse.

Runtime > Change runtime type > **T4 GPU** before running anything.

Sections:
1. GeoChat (the plan)
2. Serve it over HTTP so `backend/tools/geochat_backend.py` can call it from the laptop
3. Qwen2-VL-2B fallback (separate, run only if 1 fails)

**Things I could not verify offline, check against the repo README if output looks off:**
- Task tags. The paper uses `[grounding]` for grounded description, `[refer]` for referring expression (phrase -> box), `[identify]` for region captioning (box -> phrase). For "where is the water body" the right tag is `[refer]`, not `[grounding]`. Both are tried below.
- Box format. Paper: `{<x1><y1><x2><y2>|<θ>}`, coordinates in **0-100** of image side, θ rotation in degrees. Cell 8 prints the raw string so this gets confirmed by the model itself.
- Loader API. Written against the LLaVA-1.5 layout the repo copies (`geochat.model.builder.load_pretrained_model`). If an import fails, `ls GeoChat/geochat` and fix the module name.

## 1a. GPU check

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU"
print("torch", torch.__version__, "cuda", torch.version.cuda)

## 1b. Clone GeoChat and install

The repo pins old versions (transformers ~4.31, torch 2.0). Installing its `pyproject.toml` wholesale on Colab downgrades torch and fights the preinstalled CUDA build, which is the number one way this fails. So: clone, install only what the loader imports, keep Colab's torch.

In [ ]:
%cd /content
import os
if not os.path.isdir("GeoChat"):
    os.system("git clone --depth 1 https://github.com/mbzuai-oryx/GeoChat.git")
%cd /content/GeoChat

# 4.31.0 is the repo's own pin; newer versions removed _expand_mask that its mpt helper imports
!pip install -q "transformers==4.31.0" "accelerate==0.21.0" "bitsandbytes>=0.43" "numpy<2" "sentencepiece" "einops" "timm" "peft" "flask" "pillow" "requests" 2>&1 | tail -2

import sys
sys.path.insert(0, "/content/GeoChat")
try:
    import geochat
    print("geochat package importable from", geochat.__file__)
except ImportError as e:
    print("import failed:", e)
    print("if the repo renamed its package, look here:")
    !ls /content/GeoChat

## 1c. Download the weights

~13 GB in fp16 on the hub. Colab's disk is fine; downloading is the slow part (5-15 min). If the hub call fails (network, gated repo, rate limit) the cell says so instead of leaving a half-download.

In [ ]:
import os, shutil
from huggingface_hub import snapshot_download

WEIGHTS = "/content/geochat-7B"
try:
    path = snapshot_download("MBZUAI/geochat-7B", local_dir=WEIGHTS, local_dir_use_symlinks=False,
                             allow_patterns=["*.json", "*.bin", "*.safetensors", "*.model", "*.txt", "*.py"])
    print("weights at", path)
    !du -sh $WEIGHTS
    !ls $WEIGHTS
except Exception as e:
    print("WEIGHTS FAILED TO DOWNLOAD:", type(e).__name__, e)
    print("- check the runtime has internet (Colab free tier does)")
    print("- try again, hub 5xx errors are common; the download resumes")
    print("- or mount Drive and point WEIGHTS at a copy a teammate already downloaded")
    if os.path.isdir(WEIGHTS) and not os.listdir(WEIGHTS):
        shutil.rmtree(WEIGHTS)
    raise

## 1d. Load in 4-bit

`load_pretrained_model(..., load_4bit=True)` is the repo's own path (inherited from LLaVA), it builds a `BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=float16, bnb_4bit_quant_type="nf4")` internally. If bitsandbytes cannot see CUDA (the second classic failure) the except block prints the diagnosis and tries a manual load with our own `BitsAndBytesConfig`.

In [ ]:
import torch, traceback
from geochat.model.builder import load_pretrained_model

tokenizer = model = image_processor = None
try:
    tokenizer, model, image_processor, context_len = load_pretrained_model(
        WEIGHTS, None, "geochat-7B", load_4bit=True)
    print("loaded 4-bit via repo loader. GPU mem:", round(torch.cuda.memory_allocated()/1e9, 2), "GB")
except Exception as e:
    print("4-BIT LOAD FAILED:", type(e).__name__, e)
    traceback.print_exc()
    print("\nlikely bitsandbytes / CUDA mismatch. diagnosis:")
    !python -m bitsandbytes 2>&1 | tail -15
    print("\ntrying manual load with an explicit BitsAndBytesConfig...")
    try:
        from transformers import AutoTokenizer, BitsAndBytesConfig
        from geochat.model import GeoChatLlamaForCausalLM
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                 bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
        tokenizer = AutoTokenizer.from_pretrained(WEIGHTS, use_fast=False)
        model = GeoChatLlamaForCausalLM.from_pretrained(WEIGHTS, quantization_config=bnb, device_map="auto",
                                                       torch_dtype=torch.float16)
        vision = model.get_vision_tower()
        if not vision.is_loaded:
            vision.load_model()
        vision.to(device="cuda", dtype=torch.float16)
        image_processor = vision.image_processor
        print("manual 4-bit load worked")
    except Exception as e2:
        print("manual load also failed:", type(e2).__name__, e2)
        print("fixes to try, in order:")
        print("  1. !pip install -q -U bitsandbytes   then Runtime > Restart, rerun from 1b")
        print("  2. Runtime > Disconnect and delete runtime (gets a fresh CUDA), rerun")
        print("  3. fall back to section 3 (Qwen2-VL-2B) and tell the group")
        raise

## 1e. One test image

Swap `IMAGE_URL` for anything: an RSVQA/VRSBench tile, a Sentinel-2 true-colour crop, or a Google Maps screenshot uploaded to Colab (`IMAGE_URL = "/content/my.png"` works too). The default is a lake+shore scene from Wikimedia so the grounding query has an obvious water body to find.

In [ ]:
import io, requests
from PIL import Image

IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/4/4c/Lake_Tahoe_from_space%2C_Landsat_8.jpg"

def fetch(url):
    if url.startswith("/"):
        return Image.open(url).convert("RGB")
    r = requests.get(url, timeout=60, headers={"User-Agent": "satquery-colab"})
    r.raise_for_status()
    im = Image.open(io.BytesIO(r.content)).convert("RGB")
    im.thumbnail((1024, 1024))
    return im

try:
    image = fetch(IMAGE_URL)
except Exception as e:
    print('sample download failed, using a plain test image instead:', e)
    image = Image.new('RGB', (512, 512), (70, 110, 60))
print("image size (w, h):", image.size)
display(image.resize((min(image.width, 512), int(image.height * min(image.width, 512) / image.width))))

## 1f. Generation helper

Identical to `_load_geochat().gen` in `backend/tools/geochat_backend.py`. If you change something here that makes the model behave, port it there.

In [ ]:
from geochat.constants import DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX
from geochat.conversation import conv_templates
from geochat.mm_utils import process_images, tokenizer_image_token

def answer(image, prompt, max_new_tokens=256):
    conv = conv_templates["llava_v1"].copy()
    conv.append_message(conv.roles[0], DEFAULT_IMAGE_TOKEN + "\n" + prompt)
    conv.append_message(conv.roles[1], None)
    ids = tokenizer_image_token(conv.get_prompt(), tokenizer, IMAGE_TOKEN_INDEX,
                                return_tensors="pt").unsqueeze(0).to(model.device)
    pix = process_images([image], image_processor, model.config).to(model.device, dtype=torch.float16)
    with torch.inference_mode():
        out = model.generate(ids, images=pix, do_sample=False, max_new_tokens=max_new_tokens, use_cache=True)
    text = tokenizer.batch_decode(out[:, ids.shape[1]:], skip_special_tokens=True)[0]
    return text.strip().removesuffix("</s>").strip()

MODEL_NAME = "geochat-7B"
COORD_SCALE = 100   # GeoChat's boxes are in 0-100 of image side; confirm in 1h
print("ready")

## 1g. Plain VQA

In [ ]:
%%time
q = "What is shown in this image?"
print("Q:", q)
print("A:", answer(image, q))

## 1h. Grounding, RAW output

Two variants. `[refer]` is the phrase-to-box task and what `GeoChatGrounding` sends. `[grounding]` is grounded description (caption with boxes on everything), useful to see the format on several objects at once.

Read the printed `repr()` carefully and answer three things:
- the wrapper around a box: `{<x1><y1><x2><y2>|<θ>}`? bare `<x1><y1><x2><y2>`? something else?
- the coordinate range: all values 0-100 (normalised percent, expected) or 0-1 or pixels?
- phrase tags: is the object name wrapped in `<p>...</p>`?

`backend/tools/grounding/box_parser.py` handles the `{<..>|<..>}` + 0-100 case. If the real output differs, the regex `_TAGGED` and `COORD_SCALES` in `geochat_backend.py` are the two things to change.

In [ ]:
%%time
for prompt in ["[refer] give me the location of the water body",
               "[refer] give me the location of the lake",
               "[grounding] describe the image"]:
    raw = answer(image, prompt)
    print("PROMPT:", prompt)
    print("RAW   :", repr(raw))
    print()

import re
nums = re.findall(r"<\s*(-?\d+(?:\.\d+)?)\s*>", raw)
if nums:
    vals = [float(n) for n in nums]
    print(f"{len(vals)} numbers inside <> tags, min {min(vals)}, max {max(vals)}")
    print("=> looks like 0-100 normalised" if max(vals) <= 100 else "=> NOT 0-100, set GEOCHAT_COORD_SCALE / COORD_SCALES accordingly")
else:
    print("no <n> tags found: format differs from the paper, paste the RAW line above into the parser tests")

## 1i. Draw the boxes (sanity check that the scale guess is right)

In [ ]:
from PIL import ImageDraw

def draw_boxes(img, raw, scale=COORD_SCALE):
    w, h = img.size
    boxes = re.findall(r"<\s*(-?\d+(?:\.\d+)?)\s*>\s*<\s*(-?\d+(?:\.\d+)?)\s*>\s*<\s*(-?\d+(?:\.\d+)?)\s*>\s*<\s*(-?\d+(?:\.\d+)?)\s*>", raw)
    out = img.copy()
    d = ImageDraw.Draw(out)
    for b in boxes:
        x1, y1, x2, y2 = [float(v) for v in b]
        if scale:
            x1, x2, y1, y2 = x1/scale*w, x2/scale*w, y1/scale*h, y2/scale*h
        d.rectangle([x1, y1, x2, y2], outline="red", width=4)
    print(len(boxes), "box(es)")
    return out

raw = answer(image, "[refer] give me the location of the water body")
print(repr(raw))
display(draw_boxes(image, raw).resize((512, int(512 * image.height / image.width))))

## 2. Serve the model over HTTP

Run this after either section 1 or section 3 has defined `answer(image, prompt)` and `MODEL_NAME`. It starts Flask on port 5000 and opens a cloudflared tunnel (no account needed). Copy the printed `https://....trycloudflare.com` URL and on the laptop:

```
export GEOCHAT_ENDPOINT=https://xxxx.trycloudflare.com
export GEOCHAT_MODEL_NAME=<what this cell prints>
.venv/bin/python scripts/test_single_image_tools.py
```

Keep this Colab tab open; free Colab kills idle runtimes after ~90 min, so poke it now and then.

In [ ]:
import base64, io, threading, subprocess, time, re
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/health")
def health():
    return jsonify({"ok": True, "model": MODEL_NAME, "coord_scale": COORD_SCALE})

@app.route("/answer", methods=["POST"])
def serve_answer():
    data = request.get_json(force=True)
    img = Image.open(io.BytesIO(base64.b64decode(data["image"]))).convert("RGB")
    text = answer(img, data["prompt"], max_new_tokens=int(data.get("max_new_tokens", 256)))
    return jsonify({"text": text, "model": MODEL_NAME, "coord_scale": COORD_SCALE})

threading.Thread(target=lambda: app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False),
                 daemon=True).start()
time.sleep(2)

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
proc = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:5000", "--no-autoupdate"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(60):
    line = proc.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break
print("PUBLIC URL:", url or "not found, run `!/content/cloudflared tunnel --url http://localhost:5000` in a new cell and read it there")
print(f"export GEOCHAT_ENDPOINT={url}")
print(f"export GEOCHAT_MODEL_NAME={MODEL_NAME}")
if url:
    # the new hostname takes a few seconds to become resolvable
    for _ in range(12):
        try:
            print(requests.get(url + "/health", timeout=30).json()); break
        except Exception:
            time.sleep(5)
    else:
        print("health check did not answer yet, try the URL again in a minute")

---
# 3. Fallback: Qwen2-VL-2B-Instruct

Independent of section 1. Run 1a, 1e (for `image` and `fetch`), then this. Sets `answer`, `MODEL_NAME`, `COORD_SCALE` so section 2 serves it unchanged.

Not remote-sensing adapted; Person 4's BigEarthNet CLIP fine-tune covers the adaptation rule regardless. **Tell the group if you switch.**

In [ ]:
!pip install -q "transformers>=4.45" "accelerate" "qwen-vl-utils" 2>&1 | tail -1
import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

QWEN = "Qwen/Qwen2-VL-2B-Instruct"
qwen_model = Qwen2VLForConditionalGeneration.from_pretrained(QWEN, torch_dtype=torch.float16, device_map="auto")
qwen_proc = AutoProcessor.from_pretrained(QWEN)
print("loaded. GPU mem:", round(torch.cuda.memory_allocated()/1e9, 2), "GB")

In [ ]:
def qwen_prompt(prompt):
    # Qwen has no [refer] tag. Ask for its native box format in the text instead.
    for tag in ("[refer]", "[grounding]", "[identify]"):
        if prompt.lower().startswith(tag):
            body = prompt[len(tag):].strip()
            return (f"{body} Answer briefly, then give the bounding box of the region as "
                    f"<|box_start|>(x1,y1),(x2,y2)<|box_end|>.")
    return prompt

def answer(image, prompt, max_new_tokens=256):
    prompt = qwen_prompt(prompt)
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    chat = qwen_proc.apply_chat_template(messages, add_generation_prompt=True)
    inputs = qwen_proc(text=[chat], images=[image], return_tensors="pt").to(qwen_model.device)
    with torch.inference_mode():
        out = qwen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return qwen_proc.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()

MODEL_NAME = "Qwen2-VL-2B-Instruct"
COORD_SCALE = 1000   # Qwen2-VL grounding coords are 0-1000 of image side

print("Q: What is shown in this image?")
print("A:", answer(image, "What is shown in this image?"))
print()
# LIMITATION: Qwen2-VL was not trained with a dedicated grounding tag like GeoChat.
# We are prompting it to write a box in its answer text. Usually it does, in
# <|box_start|>(x1,y1),(x2,y2)<|box_end|> with 0-1000 coords, but this is not a
# guaranteed structured output, so box_parser.py returns [] when it does not.
raw = answer(image, "[refer] locate the water body and return its bounding box coordinates")
print("RAW grounding:", repr(raw))